Data Preprocessing:     
Load the dataset into a suitable data structure (e.g., pandas DataFrame).       
Handle missing values, if any.      
Explore the dataset to understand its structure and attributes.


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
df=pd.read_csv('anime.csv')

In [20]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  str    
 2   genre     12232 non-null  str    
 3   type      12269 non-null  str    
 4   episodes  12294 non-null  str    
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 1.3 MB


In [22]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [23]:
df=df.dropna(subset=['genre', 'rating', 'type']).reset_index(drop=True)

In [24]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
df.describe()

,anime_id,rating,members
count,12017.000000,12017.000000,1.201700e+04
mean,13638.001165,6.478264,1.834888e+04
std,11231.076675,1.023857,5.537250e+04
min,1.000000,1.670000,1.200000e+01
25%,3391.000000,5.890000,2.250000e+02
50%,9959.000000,6.570000,1.552000e+03
75%,23729.000000,7.180000,9.588000e+03
max,34519.000000,10.000000,1.013917e+06


Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).     
Convert categorical features into numerical representations if necessary.       
Normalize numerical features if required.       


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

In [30]:
tfidf=TfidfVectorizer(token_pattern=r'[^,]+')

In [31]:
genre_matrix=tfidf.fit_transform(df['genre'])

In [39]:
genre_features_count=genre_matrix.shape[1]

In [33]:
type_dummies=pd.get_dummies(df['type'])

In [34]:
type_matrix=type_dummies.astype(float).values

In [35]:
type_features_count=type_matrix.shape[1]

In [37]:
scaler=MinMaxScaler()
rating_matrix=scaler.fit_transform(df[['rating']])

In [38]:
combined=hstack([genre_matrix,type_matrix,rating_matrix])
shape=combined.shape

In [40]:
print({
    'genre_features':genre_features_count,
    'type_features':type_features_count,
    'combined_shape':shape
})

{'genre_features': 82, 'type_features': 6, 'combined_shape': (12017, 89)}


Recommendation System:

Design a function to recommend anime based on cosine similarity.        
Given a target anime, recommend a list of similar anime based on cosine similarity scores.      
Experiment with different threshold values for similarity scores to adjust the recommendation list size.        
Analyze the performance of the recommendation system and identify areas of improvement.     


In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [56]:
def get_recommendations(target_title,df,feature_matrix,threshold=0.8):
    try:
        idx=df[df['name']==target_title].index[0]
    except IndexError:
        return f"Anime '{target_title}' not found in dataset."
    target_vector=feature_matrix.tocsr()[idx]
    similarity_scores=cosine_similarity(target_vector,feature_matrix).flatten()
    df_result=df.copy()
    df_result['similarity']=similarity_scores
    recommendations=df_result[
        (df_result['similarity']>=threshold) & (df_result['name']!=target_title)
    ]
    recommendations=recommendations.sort_values(by='similarity',ascending=False)
    return recommendations[['name','genre','similarity']].head(5) #returns top 5 similarities

In [62]:
get_recommendations(target_title='Naruto',df=df,feature_matrix=combined)

,name,genre,similarity
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",0.999963
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.948696
515,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.948600
588,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.948535
1930,Dragon Ball Super,"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.947213


In [69]:
get_recommendations(target_title='Naruto',df=df,feature_matrix=combined,threshold=0.97)

,name,genre,similarity
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",0.999963


In [72]:
get_recommendations(target_title='One Piece',df=df,feature_matrix=combined,threshold=0.7)

,name,genre,similarity
2472,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",0.946536
86,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",0.939645
1192,R.O.D the TV,"Action, Adventure, Comedy, Drama, Sci-Fi, Supe...",0.931057
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.919867
515,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",0.918434


In [73]:
print("""      

The current system is fast and good at matching basic details like genres. However, it mostly
just recommends direct sequels and ignores how popular a show is or what real audiences actually 
like. To make it better, the system should track real user viewing habits (collaborative filtering) 
to see what similar fans are watching. It should also give a boost to popular shows with lots of members 
and fill in missing dataset values instead of deleting entire rows.

""")

      

The current system is fast and good at matching basic details like genres. However, it mostly
just recommends direct sequels and ignores how popular a show is or what real audiences actually 
like. To make it better, the system should track real user viewing habits (collaborative filtering) 
to see what similar fans are watching. It should also give a boost to popular shows with lots of members 
and fill in missing dataset values instead of deleting entire rows.




Interview Questions:        
1. Can you explain the difference between user-based and item-based collaborative filtering?

In [1]:
print("""      

User-based collaborative filtering recommends items to you by finding other people 
who share your tastes and suggesting what they liked. Item-based collaborative filtering 
instead focuses on the items themselves, recommending things that the overall community 
frequently likes together.

""")

      

User-based collaborative filtering recommends items to you by finding other people 
who share your tastes and suggesting what they liked. Item-based collaborative filtering 
instead focuses on the items themselves, recommending things that the overall community 
frequently likes together.




2. What is collaborative filtering, and how does it work?

In [2]:
print("""      

Collaborative filtering is a system that predicts your preferences based 
entirely on the past behaviors of a large audience, completely ignoring the 
item's actual features. It works by analyzing a massive grid of user rating 
histories to uncover hidden patterns and mathematically predict how you would 
score unseen items.

""")

      

Collaborative filtering is a system that predicts your preferences based 
entirely on the past behaviors of a large audience, completely ignoring the 
item's actual features. It works by analyzing a massive grid of user rating 
histories to uncover hidden patterns and mathematically predict how you would 
score unseen items.


